<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH13/CH13_NB05_hands_on_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13 – Hands-on lab: your own optimization project

This notebook is a **skeleton**, not a solution. It contains no executable code. Each phase has a text cell that explains its goal, a list of ideas to help you decide how to approach it, and an empty code cell marked with `# TODO` where your work begins.

You can use it for a project you have pending at work, or for any task you would simply like to try. The five phases mirror the process followed in the chapter with T (Teacher), M (Medium) and S (Small):

1. Define the task, the base model and the data
2. Create the task-specific benchmark
3. Establish the baseline
4. Create the models
5. Measure the models with the same battery

If you want to share what you build, the Chapter 13 discussion in liveBook is the place.

## Project summary

Before you start, describe your project in a few lines. Writing it down forces you to make the task explicit, and it will be useful when you share your results.

In [ ]:
# TODO: Fill in your project summary.
# Task (one or two lines):
# Base model:
# Why a specialized model instead of a frontier model (cost, latency, privacy...):
# Hardware you will use for all measurements:

## Phase 1. Define the task, the base model and the data

**Goal.** Choose a narrow, well-delimited task where a large model is overkill, pick the base model you will optimize, and obtain the data. Keep the test set untouched until Phase 5.

**Decisions you need to make**
- Which task, and why does it need a specialized model?
- Which base model, and does its chat template fit the task?
- Where do the data come from, and how do you split them?

**Ideas and possibilities**
- *Task:* routing decisions, classification, structured extraction, function calling, query rewriting, or any repetitive step in a pipeline you already run.
- *Base model:* check the model's chat template (`tokenizer.chat_template`). Some templates include native sections, such as the tools section in Qwen3, that make adapting the model to your task much easier.
- *Data sources:* a public dataset from the Hugging Face Hub, an existing dataset filtered or adapted to your domain (as we did with xLAM), traces from your own system, or synthetic data.
- *Split:* stratify by class if some cases are rare, and remove duplicate queries so the same example never appears in both train and test.

In [ ]:
# TODO: Phase 1 - load or create your dataset, choose the base model,
# inspect its chat template, and split into train and test.

## Phase 2. Create the task-specific benchmark

**Goal.** Define how you will measure whether a model does its job well on your task, and build the harness that computes it on the test set.

**Decisions you need to make**
- What does "doing it well" mean for your task, and which errors are serious?
- Is correctness binary, or do you accept some tolerance?
- How will you check that the harness measures what you think it measures?

**Ideas and possibilities**
- *Metrics:* exact match, format validity, accuracy per class, F1, or a rubric-based score if the output is free text.
- *Error taxonomy:* separating failure types (wrong format, wrong selection, wrong content) tells you much more than a single accuracy number.
- *Tolerance:* normalize case and whitespace, accept numeric values within a margin, or accept supersets of the expected text.
- *Harness validation:* run it first on answers you know are correct and on answers you know are wrong before trusting it.

In [ ]:
# TODO: Phase 2 - implement your evaluation harness and validate it
# with cases whose result you know in advance.

## Phase 3. Establish the baseline

**Goal.** Measure the original base model, before touching it, on three fronts. The results are your reference point, and in Phase 5 you will repeat exactly the same battery on every model you create.

**The three fronts**
- Your task-specific benchmark.
- General benchmarks, to know which broad capabilities you could lose.
- Efficiency, to know what it costs today to serve the model.

**Ideas and possibilities**
- *Task benchmark:* use the harness from Phase 2 on the test set.
- *General benchmarks:* use the same ones you have used throughout the book, with the same evaluation settings every time. Consider which general capabilities matter for your use case.
- *Efficiency:* tokens per second, latency per request, and memory footprint. Fix batch size, sequence length, precision and hardware, and keep them identical in Phase 5.
- *Record everything in a table* (see the comparison table at the end of this notebook).

In [ ]:
# TODO: Phase 3 - measure the base model on the task benchmark,
# on the general benchmarks, and on efficiency. Store the results.

## Phase 4. Create the models

**Goal.** Apply the techniques from the book that best fit your task and what the baseline revealed. Decide how many models you need and how much to reduce each one.

**Decisions you need to make**
- Which techniques, and which baseline problem does each one solve?
- How many models, and how much smaller should each be?
- What loss are you willing to accept? **Write it down before building the models**, so you do not choose the criterion after seeing the results.

**Ideas and possibilities**
- *Specialize:* fine-tuning with LoRA or QLoRA (Chapter 7).
- *Reduce:* depth pruning, width pruning or attention pruning, with static or data-driven selection of what to remove.
- *Recover:* knowledge distillation from the specialist or from the previous model in the family.
- *Combine:* a family of models in cascade, as we did with T, M and S, or a single optimized model if that is all your task needs.
- *Acceptance criterion:* for example, a maximum drop on the task benchmark, a maximum drop on general benchmarks, and a minimum gain in efficiency.

In [ ]:
# TODO: Phase 4 - write your acceptance criterion first (as a comment),
# then build your models.
# Acceptance criterion:

## Phase 5. Measure the models with the same battery

**Goal.** Repeat, on every model you created, exactly the battery from Phase 3: task benchmark, general benchmarks and efficiency, with the same hardware and settings. Then compare the results against the acceptance criterion from Phase 4.

**Questions to keep in mind**
- What did you gain in efficiency, and what did you pay in capabilities?
- Does each model meet the criterion you set before building it?
- Which model of the family would you put in production, and why?

**Ideas and possibilities**
- Add each model as a new row of the comparison table, so you can see at a glance what changed with respect to the baseline.
- Read small differences with care: with a small test set, one example can move the metric several points. Consider running more than one seed.
- If you used several models, compare them with each other too, not only with the baseline.

In [ ]:
# TODO: Phase 5 - run the same battery on every model you created
# and add the results to the comparison table.

## Comparison table

Fill in one row per model, starting with the baseline from Phase 3. Add or remove columns to fit your task and benchmarks.

| Model | Size (blocks / MLP expansion / parameters) | Task benchmark | General benchmark 1 | General benchmark 2 | Tokens/s | Latency (ms) | Meets criterion? |
|---|---|---|---|---|---|---|---|
| Baseline | | | | | | | – |
| Model 1 | | | | | | | |
| Model 2 | | | | | | | |
| Model 3 | | | | | | | |